# Candidate retrieval, one idea at a time

This notebook does **not** train a model. It answers one question: given the products in one session, which other products are reasonable possibilities?

We use one simple rule: if products appeared together in many past sessions, they are related.

In [ ]:
from pathlib import Path
import sys

import polars as pl

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval.covisitation import build_covisitation_matrix

For learning, we use 10,000 complete sessions from the random development sample. A session is never cut in half.

In [ ]:
events = pl.read_parquet(PROJECT_ROOT / "data/processed/train_dev.parquet")
demo_sessions = events.select("session").unique().sort("session").head(10_000)
demo_events = events.join(demo_sessions, on="session", how="inner")

print(f"Sessions: {demo_events.select('session').n_unique():,}")
print(f"Events: {len(demo_events):,}")

`aid_x -> aid_y` means that `aid_y` appeared in the same session as `aid_x`. The score is the number of different sessions that produced that pair.

In [ ]:
general_matrix = build_covisitation_matrix(demo_events, weighting="general", top_k=10)
general_matrix.head(10)

Now choose one session. Its history is what a real recommender would know when making a prediction.

In [ ]:
session_id = demo_events.select("session").unique().sort("session").item(0, 0)
history = demo_events.filter(pl.col("session") == session_id).sort("ts", descending=True)
history

Each product in the history votes for its related products. If the same product is related to several history items, we add those votes together.

In [ ]:
history_aids = history.get_column("aid").unique().to_list()

related_items = (
    general_matrix
    .filter(pl.col("aid_x").is_in(history_aids))
    .group_by("aid_y")
    .agg(pl.col("score").sum().alias("related_score"))
    .sort("related_score", descending=True)
    .head(20)
)

related_items

The final candidate list keeps both kinds of products: items already in the session and newly retrieved related items. A later model will score this small list instead of scoring every product in the catalog.

In [ ]:
candidate_list = (
    pl.concat([
        history.select(pl.col("aid").alias("candidate")).unique(),
        related_items.select(pl.col("aid_y").alias("candidate")),
    ])
    .unique()
    .with_columns(pl.col("candidate").is_in(history_aids).alias("already_in_session"))
    .join(related_items.rename({"aid_y": "candidate"}), on="candidate", how="left")
    .with_columns(pl.col("related_score").fill_null(0))
    .sort(["related_score", "already_in_session"], descending=[True, True])
)

candidate_list

You have now completed the retrieval half of a recommender:

`session history -> related products -> candidate list`

The next notebook can explain how we tell whether these candidates contain the hidden future product.